# Подбор: признаки, модели, гиперпараметры

Лидерборда нет. Значит внешней проверки, которая поймала бы нашу ошибку, не существует —
локальная оценка одновременно единственный источник правды и единственный способ себя обмануть.
Поэтому весь этот ноутбук построен вокруг одного вопроса: **как честно отличить настоящее
улучшение от везения.**

## Почему нельзя просто смотреть на два числа

Разброс нашей метрики — 0.041. Он возникает почти целиком из-за того, *какие именно куки*
попали в оценку: повезло с составом — цифра выше, не повезло — ниже.

Но когда мы сравниваем две модели, этот разброс у них **общий**: обе считались на одних и тех же
куках. Значит смотреть надо не на два числа по отдельности, а на **разницу между ними на одной и
той же выборке**.

| что смотрим | разброс |
|---|---|
| каждая модель по отдельности | ±0.041 |
| разница между ними на одних куках | в несколько раз меньше |

Механика: пересобираем выборку случайно с возвращением, и на каждой пересборке считаем метрику
**сразу для обеих** моделей. Смотрим на распределение разницы. Если новая модель выигрывает на
95% пересборок — улучшение настоящее, даже если по величине оно всего 0.015.

## Правило принятия решения

Фиксируем **до** экспериментов, чтобы потом не подгонять критерий под понравившийся результат.
Изменение принимается, только если выполнено всё:

1. средняя разница положительная;
2. новая версия выигрывает минимум в **90%** bootstrap-пересборок;
3. знак улучшения **не меняется** ни на одном из трёх повторов кросс-валидации;
4. улучшение подтверждается и на **временном** протоколе, не только на случайном.

Четыре независимых условия. Шум способен пройти одно-два, но не все четыре разом.

## Что здесь проверяется, по порядку

1. **Признаки из ресерча** — группами, каждая против текущего чемпиона.
2. **Модели** — XGBoost, LightGBM, CatBoost по отдельности.
3. **Усреднение** — по нескольким seed'ам и смесь моделей. Это не отбор, а усреднение,
   поэтому переобучиться на выборе тут невозможно в принципе.
4. **Гиперпараметры** — в конце, с готовой оснасткой под ручной перебор.

Отправная точка: 33 признака, XGBoost, **P@R≥0.7 = 0.5004**.

In [1]:
# Настройки и загрузка. Всё считается один раз, дальше переиспользуется.

import re
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from metric import precision_at_recall, recall_at_fpr

SEED = 42
np.random.seed(SEED)
DATA_DIR = Path("data")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 150)

warnings.filterwarnings("ignore", message=".*Falling back to prediction using DMatrix.*")

USE_GPU = True


def detect_device():
    if not USE_GPU:
        return "cpu"
    try:
        rs = np.random.RandomState(SEED)
        probe = XGBClassifier(n_estimators=1, max_depth=2, device="cuda",
                              tree_method="hist", verbosity=0)
        probe.fit(rs.rand(64, 4), (rs.rand(64) > 0.5).astype(int))
        used = json.loads(probe.get_booster().save_config())["learner"]["generic_param"]["device"]
        return "cuda" if str(used).startswith("cuda") else "cpu"
    except Exception as e:
        print("видеокарта недоступна, считаем на процессоре:", type(e).__name__)
        return "cpu"


DEVICE = detect_device()
print("устройство:", DEVICE)

train = pd.read_csv(DATA_DIR / "train.csv",
                    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
test = pd.read_csv(DATA_DIR / "test.csv",
                   parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
events = pd.read_csv(DATA_DIR / "events.csv.gz", parse_dates=["event_ts"])

EVENT_NAMES = [
    "search_results_view", "item_view", "photo_swipe", "seller_page_view",
    "contact_phone_show", "contact_chat_open", "contact_message_sent",
    "favorite_add", "login",
]


def normalize_platform(s):
    return s.astype("string").str.lower().replace({"iphone": "ios"})


def clip_to_window(events_raw, meta):
    """Единственная дверь к событиям. Копия из solution.ipynb, менять нельзя."""
    ev = events_raw.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id", how="inner"
    )
    keep = (ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])
    ev = ev.loc[keep].copy()
    ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort").reset_index(drop=True)
    ev["platform_norm"] = normalize_platform(ev["platform"])
    assert (ev["event_name"] == "captcha_shown").sum() == 0
    return ev


ytr = train["target"].values.astype(int)
print("train", train.shape, " test", test.shape, " доля ботов", round(ytr.mean(), 4))

устройство: cuda
train (11091, 5)  test (4909, 4)  доля ботов 0.0811


## Популяционные статистики — строго по прошлому

Популярность товара и частота user_agent считаются **накопительно**: для строки за конкретный
день берутся только события предыдущих дней.

Это та самая вторая ловушка из ресерча. Если посчитать по всем данным сразу, в популярность
товара для строки от 6 апреля попадут куки от 26-го — а их тогда не существовало. Признак от
этого становится сильно «лучше» (AUC 0.786 вместо честных 0.646), и это ровно та прибавка,
которой на проверке не будет.

Побочный эффект: у самого первого дня истории нет вообще, признак там нулевой. Так и надо.

In [2]:
# Накопительные популяционные статистики.

ev_tr_raw = clip_to_window(events, train)
ev_te_raw = clip_to_window(events, test)
ev_all = pd.concat([ev_tr_raw, ev_te_raw], ignore_index=True)

# число разных кук на товар = число пар (товар, кука): каждая кука живёт ровно в одном окне
pairs = ev_all.dropna(subset=["item_id"])[["item_id", "cookie_id", "window_start_ts"]].drop_duplicates()
per_day = pairs.groupby(["item_id", "window_start_ts"]).size().unstack(fill_value=0).sort_index(axis=1)
ITEM_POP_BEFORE = per_day.cumsum(axis=1).shift(1, axis=1).fillna(0).stack()

ua_pairs = ev_all[["user_agent", "cookie_id", "window_start_ts"]].drop_duplicates()
ua_per_day = ua_pairs.groupby(["user_agent", "window_start_ts"]).size().unstack(fill_value=0).sort_index(axis=1)
ua_before = ua_per_day.cumsum(axis=1).shift(1, axis=1).fillna(0)
UA_FREQ_BEFORE = (ua_before / ua_before.sum(axis=0).replace(0, np.nan)).stack()

print("накопительная популярность товаров:", ITEM_POP_BEFORE.shape[0], "пар (товар, день)")
print("накопительная частота user_agent  :", UA_FREQ_BEFORE.shape[0], "пар (UA, день)")

накопительная популярность товаров: 1111509 пар (товар, день)
накопительная частота user_agent  : 2960 пар (UA, день)


In [3]:
# Разбор user_agent регуляркой. Внешние библиотеки не нужны: строк всего 148 и они шаблонные.

SCRAPER_CLIENTS = ["Scrapy", "curl", "node-fetch", "urllib3", "requests", "Go-http-client"]


def ua_family(s):
    for c in SCRAPER_CLIENTS:
        if c in s:
            return "client"
    if "okhttp" in s:
        return "avito_app"
    if "YaBrowser" in s:
        return "yabrowser"
    if "Firefox" in s:
        return "firefox"
    if "Chrome" in s:
        return "chrome"
    if "Safari" in s:
        return "safari"
    return "other"


def ua_os(s):
    if "Windows NT" in s:
        return "windows"
    if "Macintosh" in s or "Mac OS X" in s:
        return "macos"
    if "X11" in s:
        return "linux_x11"
    if "Android" in s:
        return "android"
    if "iPhone" in s or "iPad" in s:
        return "ios"
    return "other"


def ua_version(s):
    for pat in [r"Chrome/(\d+)", r"Firefox/(\d+)", r"Version/(\d+)", r"Avito/(\d+)",
                r"Scrapy/(\d+)", r"curl/(\d+)", r"node-fetch/(\d+)",
                r"urllib3/(\d+)", r"requests/(\d+)", r"Go-http-client/(\d+)"]:
        m = re.search(pat, s)
        if m:
            return float(m.group(1))
    return np.nan


_ua = pd.Series(events["user_agent"].unique())
UA_INFO = pd.DataFrame({
    "user_agent": _ua.values,
    "ua_family": [ua_family(s) for s in _ua],
    "ua_os": [ua_os(s) for s in _ua],
    "ua_ver": [ua_version(s) for s in _ua],
}).set_index("user_agent")
UA_INFO["ua_is_client"] = (UA_INFO["ua_family"] == "client").astype(float)

print("разбор user_agent:")
display(UA_INFO.groupby(["ua_family", "ua_os"]).size().rename("строк UA").reset_index())

разбор user_agent:


,ua_family,ua_os,строк UA
0,avito_app,android,27
1,chrome,android,48
2,chrome,linux_x11,4
3,chrome,macos,14
4,chrome,windows,14
5,client,other,6
6,firefox,windows,14
7,other,ios,6
8,safari,macos,3
9,yabrowser,windows,12


## Признаки: база плюс кандидаты из ресерча, разложенные по группам

Одна функция считает всё сразу и помечает, какая колонка к какой группе относится. Группы нужны,
чтобы добавлять признаки **пачками**: одиночный признак почти никогда не даёт заметного сдвига, и
по нему невозможно принять решение, а группа — уже осмысленная гипотеза, которую можно принять
или отвергнуть целиком.

| группа | гипотеза одной строкой |
|---|---|
| `base` | 33 признака из `solution.ipynb`, текущий чемпион |
| `content` | сборщик ходит вширь по каталогу, покупатель сидит в своей категории и городе |
| `rhythm` | у робота ровный ритм и нет ночного провала |
| `navquery` | скрапер листает выдачу вглубь и по порядку, повторяя один запрос |
| `pointer` | у бота нет мыши, а если координаты есть — они ведут себя неестественно |
| `sequence` | порядок действий: покупатель проходит воронку, сборщик перескакивает |
| `uaplat` | чем сделан запрос и не меняется ли устройство внутри суток |

In [4]:
# Единый построитель признаков. Работает одинаково на train и test — это защита от рассинхрона.


def _ent(s):
    p = s.value_counts(normalize=True)
    return float(-(p * np.log(p)).sum()) if len(p) else np.nan


UA_FAMILIES = ["chrome", "firefox", "safari", "yabrowser", "avito_app", "client"]
UA_OSES = ["windows", "macos", "linux_x11", "android", "ios"]
PLATFORMS = ["desktop", "web", "android", "ios"]
TRANSITIONS = [
    "search_results_view>search_results_view",
    "search_results_view>item_view",
    "item_view>photo_swipe",
    "item_view>item_view",
    "item_view>contact_phone_show",
    "item_view>seller_page_view",
]


def build_features_v2(meta, events_raw, desc="признаки"):
    """Возвращает (DataFrame признаков, словарь группа -> список колонок)."""
    ev = clip_to_window(events_raw, meta)
    idx = pd.Index(meta["cookie_id"].values, name="cookie_id")
    F = pd.DataFrame(index=idx)
    groups = {}
    g = ev.groupby("cookie_id", sort=False)
    win_hours = (meta["window_end_ts"] - meta["window_start_ts"]).dt.total_seconds().values / 3600.0
    cookie_day = pd.Series(meta["window_start_ts"].values, index=meta["cookie_id"].values)

    def put(grp, name, val):
        F[name] = val.reindex(idx) if isinstance(val, pd.Series) else val
        groups.setdefault(grp, []).append(name)

    pbar = tqdm(total=7, desc=desc)

    # ---------------- BASE: те же 33 признака, что в solution.ipynb ----------------
    put("base", "n_events", g.size())
    F["n_events"] = F["n_events"].fillna(0)
    put("base", "n_items_uniq", g["item_id"].nunique())
    put("base", "n_cat_uniq", g["item_category"].nunique())
    put("base", "n_loc_uniq", g["item_location"].nunique())
    put("base", "n_query_uniq", g["search_query"].nunique())
    put("base", "events_per_hour", F["n_events"].values / win_hours)

    cnt = pd.crosstab(ev["cookie_id"], ev["event_name"]).reindex(columns=EVENT_NAMES, fill_value=0)
    cnt = cnt.reindex(idx).fillna(0.0)
    for c in EVENT_NAMES:
        put("base", "cnt_" + c, cnt[c])

    ev = ev.copy()
    ev["dt"] = g["event_ts"].diff().dt.total_seconds()
    dtd = ev.dropna(subset=["dt"])
    gd = dtd.groupby("cookie_id")["dt"]
    put("base", "dt_median", gd.median())
    put("base", "dt_min", gd.min())
    put("base", "dt_std", gd.std())
    put("base", "dt_frac_lt_1s", gd.apply(lambda s: (s < 1).mean()))
    put("base", "span_seconds", (g["event_ts"].max() - g["event_ts"].min()).dt.total_seconds())
    put("base", "n_active_hours", ev.assign(h=ev["event_ts"].dt.hour).groupby("cookie_id")["h"].nunique())

    views = F["cnt_item_view"].clip(lower=1)
    searches = F["cnt_search_results_view"].clip(lower=1)
    contacts = F["cnt_contact_phone_show"] + F["cnt_contact_chat_open"] + F["cnt_contact_message_sent"]
    put("base", "photo_per_view", F["cnt_photo_swipe"] / views)
    put("base", "contact_per_view", contacts / views)
    put("base", "view_per_search", F["cnt_item_view"] / searches)
    put("base", "max_search_page", g["search_page"].max())
    put("base", "mean_search_page", g["search_page"].mean())
    put("base", "ptr_frac", g["pointer_x"].apply(lambda s: s.notna().mean()))
    put("base", "ptr_x_nunique", g["pointer_x"].nunique())
    xy = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    xy["_xy"] = xy["pointer_x"].astype(int).astype(str) + "_" + xy["pointer_y"].astype(int).astype(str)
    put("base", "ptr_xy_nunique", xy.groupby("cookie_id")["_xy"].nunique())
    F["ptr_xy_nunique"] = F["ptr_xy_nunique"].fillna(0.0)
    put("base", "n_platform_norm", g["platform_norm"].nunique())
    put("base", "n_user_agent", g["user_agent"].nunique())
    put("base", "cookie_age_days",
        (meta["window_end_ts"] - meta["cookie_created_at"]).dt.total_seconds().values / 86400.0)
    put("base", "window_dow", meta["window_start_ts"].dt.dayofweek.values.astype(float))
    pbar.update(1)

    # ---------------- CONTENT ----------------
    evc = ev.dropna(subset=["item_category"])
    put("content", "cat_entropy", evc.groupby("cookie_id")["item_category"].apply(_ent))
    put("content", "cat_top_frac",
        evc.groupby("cookie_id")["item_category"].apply(lambda s: s.value_counts(normalize=True).max()))
    evc = evc.copy()
    evc["prev"] = evc.groupby("cookie_id")["item_category"].shift()
    put("content", "cat_switch_frac",
        evc.assign(sw=((evc["prev"].notna()) & (evc["prev"] != evc["item_category"])).astype(float))
           .groupby("cookie_id")["sw"].mean())
    evl = ev.dropna(subset=["item_location"])
    put("content", "loc_entropy", evl.groupby("cookie_id")["item_location"].apply(_ent))
    put("content", "loc_top_frac",
        evl.groupby("cookie_id")["item_location"].apply(lambda s: s.value_counts(normalize=True).max()))
    evi = ev.dropna(subset=["item_id"])
    gi = evi.groupby("cookie_id")
    put("content", "item_repeat_frac", 1 - gi["item_id"].nunique() / gi.size())
    _ik = pd.MultiIndex.from_arrays([evi["item_id"].values, evi["window_start_ts"].values])
    put("content", "item_pop_mean",
        evi.assign(pop=ITEM_POP_BEFORE.reindex(_ik).values).groupby("cookie_id")["pop"].mean())
    stt = pd.crosstab(ev["cookie_id"], ev["seller_type"], normalize="index")
    put("content", "seller_frac_private",
        stt["private"] if "private" in stt.columns else pd.Series(0.0, index=stt.index))
    put("content", "seller_known_frac", g["seller_type"].apply(lambda s: s.notna().mean()))
    pbar.update(1)

    # ---------------- RHYTHM ----------------
    put("rhythm", "dt_mean", gd.mean())
    put("rhythm", "dt_max", gd.max())
    put("rhythm", "dt_p10", gd.quantile(0.10))
    put("rhythm", "dt_p90", gd.quantile(0.90))
    put("rhythm", "dt_frac_lt_5s", gd.apply(lambda s: (s < 5).mean()))
    F["dt_cv"] = F["dt_std"] / F["dt_mean"].clip(lower=1e-9)
    groups["rhythm"].append("dt_cv")
    ev["new_sess"] = ev["dt"].isna() | (ev["dt"] > 1800)
    ev["sess_id"] = ev.groupby("cookie_id")["new_sess"].cumsum()
    sess = ev.groupby(["cookie_id", "sess_id"]).size().rename("n").reset_index()
    put("rhythm", "n_sessions", sess.groupby("cookie_id")["sess_id"].max())
    put("rhythm", "sess_events_mean", sess.groupby("cookie_id")["n"].mean())
    put("rhythm", "sess_events_max", sess.groupby("cookie_id")["n"].max())
    put("rhythm", "hour_entropy", ev.assign(h=ev["event_ts"].dt.hour).groupby("cookie_id")["h"].apply(_ent))
    put("rhythm", "night_frac",
        ev.assign(n=(ev["event_ts"].dt.hour < 6).astype(float)).groupby("cookie_id")["n"].mean())
    pbar.update(1)

    # ---------------- NAVQUERY ----------------
    srch = ev[ev["event_name"] == "search_results_view"].copy()
    gs = srch.groupby("cookie_id")
    put("navquery", "page_gt5_frac", gs["search_page"].apply(lambda s: (s > 5).mean()))
    put("navquery", "page_gt10_frac", gs["search_page"].apply(lambda s: (s > 10).mean()))
    put("navquery", "page_std", gs["search_page"].std())
    srch["prev_page"] = gs["search_page"].shift()
    put("navquery", "page_step_plus1_frac",
        srch.assign(st=((srch["search_page"] - srch["prev_page"]) == 1).astype(float))
            .groupby("cookie_id")["st"].mean())
    n_srch = gs.size()
    n_q = gs["search_query"].nunique()
    put("navquery", "q_repeat_frac", 1 - n_q / n_srch.clip(lower=1))
    put("navquery", "q_len_mean", gs["search_query"].apply(lambda s: s.dropna().str.len().mean()))
    pbar.update(1)

    # ---------------- POINTER ----------------
    pxy = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    pxy["dist"] = np.sqrt(pxy.groupby("cookie_id")["pointer_x"].diff() ** 2
                          + pxy.groupby("cookie_id")["pointer_y"].diff() ** 2)
    pxy["_xy"] = pxy["pointer_x"].astype(int).astype(str) + "_" + pxy["pointer_y"].astype(int).astype(str)
    gp = pxy.groupby("cookie_id")
    put("pointer", "ptr_x_std", gp["pointer_x"].std())
    put("pointer", "ptr_y_std", gp["pointer_y"].std())
    put("pointer", "ptr_dist_mean", gp["dist"].mean())
    put("pointer", "ptr_dist_std", gp["dist"].std())
    put("pointer", "ptr_repeat_frac", 1 - gp["_xy"].nunique() / gp.size())
    pbar.update(1)

    # ---------------- SEQUENCE ----------------
    sq = ev[["cookie_id", "event_name"]].copy()
    sq["nxt"] = sq.groupby("cookie_id")["event_name"].shift(-1)
    sq = sq.dropna(subset=["nxt"])
    sq["pair"] = sq["event_name"] + ">" + sq["nxt"]
    tot = sq.groupby("cookie_id").size()
    pv = pd.crosstab(sq["cookie_id"], sq["pair"])
    for pair in TRANSITIONS:
        col = pv[pair] if pair in pv.columns else pd.Series(0.0, index=pv.index)
        put("sequence", "tr_" + pair.replace(">", "_to_"), col / tot)
    put("sequence", "funnel_depth",
        (F["cnt_search_results_view"] > 0).astype(int) + (F["cnt_item_view"] > 0).astype(int)
        + (F["cnt_photo_swipe"] > 0).astype(int) + (contacts > 0).astype(int))
    pbar.update(1)

    # ---------------- UA + PLATFORM ----------------
    ua_ck = ev.groupby("cookie_id")["user_agent"].agg(lambda s: s.mode().iat[0])
    uj = UA_INFO.reindex(ua_ck.values)
    put("uaplat", "ua_is_client", pd.Series(uj["ua_is_client"].values, index=ua_ck.index))
    put("uaplat", "ua_ver", pd.Series(uj["ua_ver"].values, index=ua_ck.index))
    _uk = pd.MultiIndex.from_arrays([ua_ck.values, cookie_day.reindex(ua_ck.index).values])
    put("uaplat", "ua_freq", pd.Series(UA_FREQ_BEFORE.reindex(_uk).values, index=ua_ck.index))
    for fam in UA_FAMILIES:
        put("uaplat", "ua_fam_" + fam,
            pd.Series((uj["ua_family"].values == fam).astype(float), index=ua_ck.index))
    for os_ in UA_OSES:
        put("uaplat", "ua_os_" + os_,
            pd.Series((uj["ua_os"].values == os_).astype(float), index=ua_ck.index))
    pl = pd.crosstab(ev["cookie_id"], ev["platform_norm"], normalize="index")
    for p in PLATFORMS:
        put("uaplat", "plat_frac_" + p, pl[p] if p in pl.columns else pd.Series(0.0, index=pl.index))
    ev["prev_plat"] = ev.groupby("cookie_id")["platform_norm"].shift()
    put("uaplat", "plat_switch_frac",
        ev.assign(sw=((ev["prev_plat"].notna()) & (ev["prev_plat"] != ev["platform_norm"])).astype(float))
          .groupby("cookie_id")["sw"].mean())
    pbar.update(1)
    pbar.close()

    # счётчики — нули, всё остальное оставляем NaN: бустинг сам разберётся
    zero = ["n_events", "events_per_hour", "ptr_xy_nunique"] + ["cnt_" + c for c in EVENT_NAMES]
    F[zero] = F[zero].fillna(0.0)
    return F.reset_index(drop=True), groups


Xtr, GROUPS = build_features_v2(train, events, "признаки train")
Xte, _ = build_features_v2(test, events, "признаки test")

assert list(Xtr.columns) == list(Xte.columns)
assert len(Xtr) == len(train) and len(Xte) == len(test)

BASE = GROUPS["base"]
EXTRA_GROUPS = {k: v for k, v in GROUPS.items() if k != "base"}

print("\nвсего признаков:", Xtr.shape[1])
print("база:", len(BASE))
for k, v in EXTRA_GROUPS.items():
    print(f"  {k:10s} +{len(v):>2}: {', '.join(v)}")

признаки train:   0%|          | 0/7 [00:00<?, ?it/s]

признаки test:   0%|          | 0/7 [00:00<?, ?it/s]


всего признаков: 90
база: 33
  content    + 9: cat_entropy, cat_top_frac, cat_switch_frac, loc_entropy, loc_top_frac, item_repeat_frac, item_pop_mean, seller_frac_private, seller_known_frac
  rhythm     +11: dt_mean, dt_max, dt_p10, dt_p90, dt_frac_lt_5s, dt_cv, n_sessions, sess_events_mean, sess_events_max, hour_entropy, night_frac
  navquery   + 6: page_gt5_frac, page_gt10_frac, page_std, page_step_plus1_frac, q_repeat_frac, q_len_mean
  pointer    + 5: ptr_x_std, ptr_y_std, ptr_dist_mean, ptr_dist_std, ptr_repeat_frac
  sequence   + 7: tr_search_results_view_to_search_results_view, tr_search_results_view_to_item_view, tr_item_view_to_photo_swipe, tr_item_view_to_item_view, tr_item_view_to_contact_phone_show, tr_item_view_to_seller_page_view, funnel_depth
  uaplat     +19: ua_is_client, ua_ver, ua_freq, ua_fam_chrome, ua_fam_firefox, ua_fam_safari, ua_fam_yabrowser, ua_fam_avito_app, ua_fam_client, ua_os_windows, ua_os_macos, ua_os_linux_x11, ua_os_android, ua_os_ios, plat_frac_desk

## Оснастка: как мы сравниваем

Три инструмента, дальше всё крутится вокруг них.

**`run_cv`** — кросс-валидация 5 частей × 3 повтора. Каждый повтор сам по себе покрывает все
строки, поэтому на выходе получаются **три независимых OOF-вектора** плюс их среднее. Три вектора
достаются бесплатно и дают проверку устойчивости: если улучшение настоящее, оно должно быть видно
во всех трёх, а не в одном удачном.

**`run_time`** — обучение на прошлом, проверка на будущем. Вторая, независимая точка зрения.

**`paired_bootstrap`** — главный инструмент. Пересобирает выборку 500 раз и на каждой пересборке
считает метрику **обеих** моделей сразу. Возвращает распределение разницы.

**`verdict`** — применяет те самые четыре условия и выносит решение. Ничего не решаем на глаз.

In [5]:
# Фабрики моделей и оснастка сравнения.

def make_xgb(seed=SEED):
    return XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        device=DEVICE, tree_method="hist",
        eval_metric="logloss", random_state=seed, verbosity=0, n_jobs=-1,
    )


def make_lgbm(seed=SEED):
    # LightGBM всегда на процессоре: колёса с pypi собраны без поддержки видеокарты
    return LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=seed, n_jobs=-1, verbose=-1,
    )


def make_cat(seed=SEED):
    # rsm (доля колонок на дерево) на видеокарте CatBoost не поддерживает — падает с ошибкой.
    # На GPU просто не задаём его, на процессоре оставляем как у остальных моделей.
    p = dict(iterations=500, learning_rate=0.05, depth=6,
             random_seed=seed, verbose=0, allow_writing_files=False)
    if DEVICE == "cuda":
        p["task_type"] = "GPU"
    else:
        p["task_type"] = "CPU"
        p["rsm"] = 0.8
    return CatBoostClassifier(**p)


class Bagged:
    """Обучает одну и ту же модель на нескольких seed'ах и усредняет предсказания.

    Это НЕ отбор: мы ничего не выбираем, а усредняем. Переобучиться на выборе тут
    невозможно в принципе — просто гасим случайность обучения.
    """

    def __init__(self, factory, seeds):
        self.factory, self.seeds = factory, seeds

    def fit(self, X, y):
        self.models_ = [self.factory(s).fit(X, y) for s in self.seeds]
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models_], axis=0)


N_SPLITS, N_REPEATS = 5, 3

TIME_SPEC = [
    ("2026-04-06", "2026-04-13", "2026-04-14", "2026-04-15"),
    ("2026-04-06", "2026-04-15", "2026-04-16", "2026-04-17"),
    ("2026-04-06", "2026-04-17", "2026-04-18", "2026-04-19"),
]


def run_cv(make_model, cols, desc=""):
    """Случайный протокол. Возвращает (среднее OOF, матрицу OOF по повторам)."""
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    oof_rep = np.full((N_REPEATS, len(ytr)), np.nan)
    for k, (tr, va) in enumerate(tqdm(list(rskf.split(Xtr, ytr)), desc=desc, leave=False)):
        m = make_model()
        m.fit(Xtr.iloc[tr][cols], ytr[tr])
        oof_rep[k // N_SPLITS, va] = m.predict_proba(Xtr.iloc[va][cols])[:, 1]
    return oof_rep.mean(axis=0), oof_rep


def run_time(make_model, cols):
    """Временной протокол: учимся на прошлом, проверяемся на будущем."""
    day = train["window_start_ts"].dt.normalize()
    oof = np.full(len(ytr), np.nan)
    for a, b, c, d in TIME_SPEC:
        tr = np.where(((day >= a) & (day <= b)).values)[0]
        va = np.where(((day >= c) & (day <= d)).values)[0]
        m = make_model()
        m.fit(Xtr.iloc[tr][cols], ytr[tr])
        oof[va] = m.predict_proba(Xtr.iloc[va][cols])[:, 1]
    return oof


def pr(y, s):
    m = ~np.isnan(s)
    return precision_at_recall(y[m], s[m])


def paired_bootstrap(a, b, n_boot=500, seed=SEED):
    """Распределение разницы метрик двух моделей НА ОДНИХ И ТЕХ ЖЕ пересобранных данных.

    Шум от состава выборки у обеих моделей общий и при вычитании почти полностью уходит,
    поэтому разницу видно гораздо точнее, чем каждую метрику по отдельности.
    """
    mask = ~(np.isnan(a) | np.isnan(b))
    yy, aa, bb = ytr[mask], a[mask], b[mask]
    rng = np.random.default_rng(seed)
    n = len(yy)
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        ys = yy[idx]
        d[i] = precision_at_recall(ys, bb[idx]) - precision_at_recall(ys, aa[idx])
    return d


REGISTRY = []


def evaluate(make_model, cols, name):
    """Полный прогон кандидата по обоим протоколам."""
    t0 = time.time()
    oof, oof_rep = run_cv(make_model, cols, desc=name)
    oof_t = run_time(make_model, cols)
    cand = {
        "name": name, "cols": list(cols), "make": make_model,
        "oof": oof, "oof_rep": oof_rep, "oof_time": oof_t,
        "pr": pr(ytr, oof), "pr_time": pr(ytr, oof_t),
        "auc": roc_auc_score(ytr, oof), "n_feat": len(cols),
        "sec": time.time() - t0,
    }
    REGISTRY.append({k: cand[k] for k in ["name", "n_feat", "pr", "pr_time", "auc", "sec"]})
    print(f"{name:28s} признаков {len(cols):>3}  P@R {cand['pr']:.4f}  "
          f"время-протокол {cand['pr_time']:.4f}  AUC {cand['auc']:.4f}  ({cand['sec']:.0f} c)")
    return cand


def verdict(champ, chal, n_boot=500, min_win=0.90, show=True):
    """Те самые четыре условия. Возвращает True, только если выполнены все."""
    d = paired_bootstrap(champ["oof"], chal["oof"], n_boot)
    win = float((d > 0).mean())
    reps = [pr(ytr, chal["oof_rep"][r]) - pr(ytr, champ["oof_rep"][r])
            for r in range(champ["oof_rep"].shape[0])]
    dt_time = chal["pr_time"] - champ["pr_time"]

    cond = {
        "1. средняя разница > 0": float(d.mean()) > 0,
        f"2. выигрывает в >= {min_win:.0%} пересборок": win >= min_win,
        "3. знак одинаков на всех повторах": all(x > 0 for x in reps),
        "4. подтверждено временным протоколом": dt_time > 0,
    }
    ok = all(cond.values())

    if show:
        print(f"\n  {chal['name']}  против  {champ['name']}")
        print(f"    разница метрики: {chal['pr'] - champ['pr']:+.4f} "
              f"(было {champ['pr']:.4f}, стало {chal['pr']:.4f})")
        print(f"    bootstrap: среднее {d.mean():+.4f}, выигрывает в {win:.1%} пересборок, "
              f"интервал 5-95%: [{np.percentile(d, 5):+.4f}, {np.percentile(d, 95):+.4f}]")
        print(f"    по повторам CV: {', '.join(f'{x:+.4f}' for x in reps)}")
        print(f"    временной протокол: {dt_time:+.4f}")
        for k, v in cond.items():
            print(f"    [{'OK' if v else '--'}] {k}")
        print(f"    ВЕРДИКТ: {'ПРИНЯТО' if ok else 'отклонено'}")
    return ok


print("оснастка готова. один прогон кандидата — 18 обучений, примерно полминуты")

оснастка готова. один прогон кандидата — 18 обучений, примерно полминуты


## Шаг 1. Признаки из ресерча, группа за группой

Начинаем с базы в 33 признака — это текущий чемпион, `solution.ipynb`, P@R = 0.5004.

Дальше по очереди пробуем добавить каждую группу к **текущему чемпиону**. Если группа принята —
она остаётся и следующая группа проверяется уже вместе с ней. Если отклонена — откатываем и идём
дальше. Это называется жадный отбор вперёд, и он честен ровно настолько, насколько честен критерий
принятия, — а критерий у нас записан заранее и состоит из четырёх условий.

Порядок групп — по силе одномерного скрининга из `research.ipynb`, от сильной к слабой.

In [6]:
# Жадный отбор групп признаков. Займёт несколько минут.

champion = evaluate(make_xgb, BASE, "база (33 признака)")

GROUP_ORDER = ["content", "rhythm", "navquery", "pointer", "sequence", "uaplat"]
accepted, rejected = [], []

for gname in GROUP_ORDER:
    cols = champion["cols"] + [c for c in EXTRA_GROUPS[gname] if c not in champion["cols"]]
    chal = evaluate(make_xgb, cols, f"+ {gname}")
    if verdict(champion, chal):
        champion = chal
        champion["name"] = f"чемпион после +{gname}"
        accepted.append(gname)
    else:
        rejected.append(gname)

print("\n" + "=" * 70)
print("принятые группы:", accepted if accepted else "ни одной")
print("отклонённые    :", rejected if rejected else "нет")
print(f"итог: {len(champion['cols'])} признаков, P@R = {champion['pr']:.4f} "
      f"(было 0.5004 на 33 признаках)")
print("=" * 70)

FEATURES_BEST = champion["cols"]

база (33 признака):   0%|          | 0/15 [00:00<?, ?it/s]

база (33 признака)           признаков  33  P@R 0.5004  время-протокол 0.4611  AUC 0.8946  (25 c)


+ content:   0%|          | 0/15 [00:00<?, ?it/s]

+ content                    признаков  42  P@R 0.6435  время-протокол 0.6484  AUC 0.9086  (24 c)

  + content  против  база (33 признака)
    разница метрики: +0.1431 (было 0.5004, стало 0.6435)
    bootstrap: среднее +0.1418, выигрывает в 100.0% пересборок, интервал 5-95%: [+0.0950, +0.1855]
    по повторам CV: +0.1347, +0.1325, +0.1470
    временной протокол: +0.1873
    [OK] 1. средняя разница > 0
    [OK] 2. выигрывает в >= 90% пересборок
    [OK] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: ПРИНЯТО


+ rhythm:   0%|          | 0/15 [00:00<?, ?it/s]

...

+ pointer                    признаков  47  P@R 0.8098  время-протокол 0.7205  AUC 0.9326  (23 c)

  + pointer  против  чемпион после +content
    разница метрики: +0.1663 (было 0.6435, стало 0.8098)
    bootstrap: среднее +0.1589, выигрывает в 100.0% пересборок, интервал 5-95%: [+0.1150, +0.2049]
    по повторам CV: +0.1664, +0.1906, +0.1547
    временной протокол: +0.0720
    [OK] 1. средняя разница > 0
    [OK] 2. выигрывает в >= 90% пересборок
    [OK] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: ПРИНЯТО


+ sequence:   0%|          | 0/15 [00:00<?, ?it/s]

+ sequence                   признаков  54  P@R 0.8056  время-протокол 0.7615  AUC 0.9321  (25 c)

  + sequence  против  чемпион после +pointer
    разница метрики: -0.0041 (было 0.8098, стало 0.8056)
    bootstrap: среднее +0.0033, выигрывает в 56.8% пересборок, интервал 5-95%: [-0.0172, +0.0274]
    по повторам CV: -0.0302, +0.0090, +0.0151
    временной протокол: +0.0410
    [OK] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: отклонено


+ uaplat:   0%|          | 0/15 [00:00<?, ?it/s]

+ uaplat                     признаков  66  P@R 0.8108  время-протокол 0.7733  AUC 0.9332  (25 c)

  + uaplat  против  чемпион после +pointer
    разница метрики: +0.0010 (было 0.8098, стало 0.8108)
    bootstrap: среднее +0.0056, выигрывает в 58.4% пересборок, интервал 5-95%: [-0.0218, +0.0370]
    по повторам CV: +0.0069, +0.0072, +0.0342
    временной протокол: +0.0528
    [OK] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [OK] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: отклонено

принятые группы: ['content', 'pointer']
отклонённые    : ['rhythm', 'navquery', 'sequence', 'uaplat']
итог: 47 признаков, P@R = 0.8098 (было 0.5004 на 33 признаках)


## Шаг 2. Три модели на победившем наборе

Теперь фиксируем признаки и меняем модель. XGBoost, LightGBM, CatBoost — одинаковые разумные
настройки, никакого подбора.

Смысл не только в том, чтобы выбрать лучшую. Три модели ошибаются по-разному, и это пригодится на
следующем шаге: если усреднить их предсказания, часть ошибок взаимно погасится.

In [7]:
# Сравниваем три модели на одном наборе признаков.

cand_xgb = champion if champion["cols"] == FEATURES_BEST else evaluate(make_xgb, FEATURES_BEST, "XGBoost")
cand_xgb["name"] = "XGBoost"
cand_lgb = evaluate(make_lgbm, FEATURES_BEST, "LightGBM")
cand_cat = evaluate(make_cat, FEATURES_BEST, "CatBoost")

print()
for c in [cand_lgb, cand_cat]:
    verdict(cand_xgb, c)

LightGBM:   0%|          | 0/15 [00:00<?, ?it/s]

LightGBM                     признаков  47  P@R 0.8036  время-протокол 0.7576  AUC 0.9347  (8 c)


CatBoost:   0%|          | 0/15 [00:00<?, ?it/s]

CatBoost                     признаков  47  P@R 0.7990  время-протокол 0.7886  AUC 0.9367  (346 c)


  LightGBM  против  XGBoost
    разница метрики: -0.0062 (было 0.8098, стало 0.8036)
    bootstrap: среднее +0.0045, выигрывает в 61.8% пересборок, интервал 5-95%: [-0.0156, +0.0271]
    по повторам CV: +0.0099, -0.0107, +0.0024
    временной протокол: +0.0371
    [OK] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: отклонено

  CatBoost  против  XGBoost
    разница метрики: -0.0108 (было 0.8098, стало 0.7990)
    bootstrap: среднее -0.0019, выигрывает в 42.4% пересборок, интервал 5-95%: [-0.0285, +0.0275]
    по повторам CV: -0.0075, +0.0020, +0.0160
    временной протокол: +0.0682
    [--] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: отклонено


## Шаг 3. Усреднение — самый безопасный способ улучшиться

Два вида усреднения, и оба принципиально отличаются от всего предыдущего.

**По seed'ам.** Одна и та же модель, обученная с разным `random_state`, даёт немного разные
предсказания — это случайность обучения. Усредняем пять таких моделей, случайность гасится.

**По моделям.** XGBoost, LightGBM и CatBoost усредняем **по рангам**, а не по вероятностям.
Ранги — потому что метрику определяет только порядок, а шкалы вероятностей у трёх библиотек
разные, и усреднять их напрямую некорректно.

Ключевое: здесь мы **ничего не выбираем**. А раз нет выбора — нет и отбора по шуму, той самой
ловушки, из-за которой опасен перебор гиперпараметров. Поэтому усреднение почти всегда безопасно.

In [8]:
# Усреднение по seed'ам и смесь трёх моделей.

cand_bag = evaluate(lambda: Bagged(make_xgb, [42, 43, 44, 45, 46]), FEATURES_BEST,
                    "XGBoost x5 seed")
verdict(cand_xgb, cand_bag)


def rank_avg(*oofs):
    """Усреднение по рангам: метрике важен только порядок, а не шкала."""
    return np.mean([pd.Series(o).rank(pct=True).values for o in oofs], axis=0)


mix = {
    "name": "смесь XGB+LGBM+Cat (ранги)",
    "cols": FEATURES_BEST,
    "oof": rank_avg(cand_xgb["oof"], cand_lgb["oof"], cand_cat["oof"]),
    "oof_rep": np.stack([rank_avg(cand_xgb["oof_rep"][r], cand_lgb["oof_rep"][r], cand_cat["oof_rep"][r])
                         for r in range(N_REPEATS)]),
    "oof_time": rank_avg(cand_xgb["oof_time"], cand_lgb["oof_time"], cand_cat["oof_time"]),
}
mix["pr"] = pr(ytr, mix["oof"])
mix["pr_time"] = pr(ytr, mix["oof_time"])
mix["n_feat"] = len(FEATURES_BEST)
REGISTRY.append({"name": mix["name"], "n_feat": mix["n_feat"], "pr": mix["pr"],
                 "pr_time": mix["pr_time"], "auc": roc_auc_score(ytr, mix["oof"]), "sec": 0.0})
print(f"\n{mix['name']:28s} P@R {mix['pr']:.4f}  время-протокол {mix['pr_time']:.4f}")
verdict(cand_xgb, mix)

XGBoost x5 seed:   0%|          | 0/15 [00:00<?, ?it/s]

XGBoost x5 seed              признаков  47  P@R 0.7878  время-протокол 0.7545  AUC 0.9331  (122 c)

  XGBoost x5 seed  против  XGBoost
    разница метрики: -0.0220 (было 0.8098, стало 0.7878)
    bootstrap: среднее -0.0061, выигрывает в 28.4% пересборок, интервал 5-95%: [-0.0254, +0.0086]
    по повторам CV: -0.0029, +0.0030, +0.0151
    временной протокол: +0.0341
    [--] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: отклонено

смесь XGB+LGBM+Cat (ранги)   P@R 0.7955  время-протокол 0.7806

  смесь XGB+LGBM+Cat (ранги)  против  XGBoost
    разница метрики: -0.0143 (было 0.8098, стало 0.7955)
    bootstrap: среднее +0.0056, выигрывает в 71.2% пересборок, интервал 5-95%: [-0.0175, +0.0270]
    по повторам CV: +0.0030, +0.0121, +0.0189
    временной протокол: +0.0601
    [OK] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [OK] 3. знак одинаков 

False

In [9]:
# Фиксируем чемпиона после трёх шагов. Дальше гиперпараметры сравниваются с ним.

CANDIDATES = {c["name"]: c for c in [cand_xgb, cand_lgb, cand_cat, cand_bag, mix]}
board = pd.DataFrame([{"кандидат": k, "признаков": v["n_feat"],
                       "P@R случайный": round(v["pr"], 4),
                       "P@R временной": round(v["pr_time"], 4)}
                      for k, v in CANDIDATES.items()]).sort_values("P@R случайный", ascending=False)
print("Все кандидаты после шагов 1-3:")
display(board.reset_index(drop=True))

# Берём лучшего по случайному протоколу, но осознанно: разрыв между соседями может быть
# в пределах шума. Если верхние два отличаются меньше чем на 0.01 — выбирай тот, что проще.
CHAMPION_FINAL = CANDIDATES[board.iloc[0]["кандидат"]]
print(f"\nчемпион: {CHAMPION_FINAL['name']}  P@R = {CHAMPION_FINAL['pr']:.4f}")
print(f"разрыв со вторым местом: {board.iloc[0]['P@R случайный'] - board.iloc[1]['P@R случайный']:+.4f}")

Все кандидаты после шагов 1-3:


,кандидат,признаков,P@R случайный,P@R временной
0,XGBoost,47,0.8098,0.7205
1,LightGBM,47,0.8036,0.7576
2,CatBoost,47,0.7990,0.7886
3,смесь XGB+LGBM+Cat (ранги),47,0.7955,0.7806
4,XGBoost x5 seed,47,0.7878,0.7545



чемпион: XGBoost  P@R = 0.8098
разрыв со вторым местом: +0.0062


## Шаг 4. Гиперпараметры — твоя часть

Здесь надо крутить руками и осознанно, поэтому оставляю инструмент, а не готовый ответ.

### Сначала — почему нельзя перебирать много

Каждая конфигурация даёт зашумлённую оценку. Если взять **лучшую из N**, ожидаемое завышение
примерно `σ · √(2·ln N)`, где σ — разброс метрики:

| N конфигураций | завышение на пустом месте |
|---|---|
| 10 | ≈ 0.09 |
| 50 | ≈ 0.11 |
| 200 | ≈ 0.13 |

Перебрав 50 вариантов, вы увидите прибавку около +0.11, **даже если все конфигурации одинаковы по
качеству**. Лидерборда, который бы это опроверг, у нас нет.

Важно: эта арифметика касается сравнения **абсолютных** чисел. Наше парное сравнение устроено
иначе — оно смотрит на разницу на одних и тех же данных, и потому гораздо устойчивее. Но
защищённость не бесконечна: если прогнать сотню конфигураций и взять ту, что случайно прошла все
четыре условия, отбор по шуму вернётся. Поэтому держим **не больше 20-30 попыток** и не
переигрываем проигравших.

### Что крутить и в каком порядке

1. `learning_rate` вместе с `n_estimators` — это одна ручка, а не две: уменьшил шаг, увеличь число деревьев;
2. `max_depth` — глубина; при 8% положительных глубокие деревья быстро начинают запоминать;
3. `min_child_weight` — сколько наблюдений должно остаться в листе; прямая защита от запоминания;
4. `subsample`, `colsample_bytree` — доля строк и колонок на дерево;
5. `reg_lambda` — регуляризация.

### Как пользоваться

Меняешь параметры в вызове `try_params(...)` и запускаешь ячейку. Она сама прогонит оба протокола,
сделает парное сравнение с чемпионом и вынесет вердикт по четырём условиям. Один прогон — примерно
полминуты.

Веди список того, что пробовал, прямо в ячейке ниже — иначе через десять попыток забудешь.

In [10]:
# Журналы попыток. Заводятся ОДИН раз: повторный запуск ячейки их не стирает,
# поэтому история перебора копится между прогонами.

try:
    TRIED_XGB
except NameError:
    TRIED_XGB = []
try:
    TRIED_CAT
except NameError:
    TRIED_CAT = []

# если в ядре остался старый общий список — переносим его в XGBoost-журнал
if "TRIED" in globals() and TRIED and not TRIED_XGB:
    TRIED_XGB.extend(TRIED)
    print("перенесено из старого TRIED:", len(TRIED), "записей")

print("XGBoost попыток:", len(TRIED_XGB), "| CatBoost попыток:", len(TRIED_CAT))


XGBoost попыток: 0 | CatBoost попыток: 0


In [19]:
# Ручной перебор. Меняй аргументы и запускай ячейку заново.


def try_params(**kw):
    """Прогоняет одну конфигурацию и сравнивает её с чемпионом по всем четырём условиям."""
    params = dict(n_estimators=500, learning_rate=0.05, max_depth=6,
                  subsample=0.8, colsample_bytree=0.8, min_child_weight=1, reg_lambda=1.0)
    params.update(kw)
    label = "params: " + ", ".join(f"{k}={v}" for k, v in kw.items()) if kw else "params: дефолт"

    def factory():
        return XGBClassifier(**params, device=DEVICE, tree_method="hist",
                             eval_metric="logloss", random_state=SEED, verbosity=0, n_jobs=-1)

    cand = evaluate(factory, FEATURES_BEST, label)
    ok = verdict(CHAMPION_FINAL, cand)
    TRIED_XGB.append({**params, "P@R": round(cand["pr"], 4),
                      "P@R время": round(cand["pr_time"], 4),
                      "сек всего": round(cand["sec"], 1),
                      "сек/обучение": round(cand["sec"] / 18, 2), "принято": ok})
    return cand


# Примеры для начала — раскомментируй по одному, не все сразу.
#try_params(max_depth=4 <<
#try_params(max_depth=10)
try_params(learning_rate=0.02, n_estimators=1500, min_child_weight=20)
#try_params(min_child_weight=20)
# try_params(subsample=0.6, colsample_bytree=0.6)
# try_params(reg_lambda=20.0)

if TRIED_XGB:
    print("\nXGBoost — все попытки за все запуски:")
    display(pd.DataFrame(TRIED_XGB).sort_values("P@R время", ascending=False))
else:
    print("пока ничего не пробовал — раскомментируй одну строку выше и запусти ячейку")

params: learning_rate=0.02, n_estimators=1500, min_child_weight=20:   0%|          | 0/15 [00:00<?, ?it/s]

params: learning_rate=0.02, n_estimators=1500, min_child_weight=20 признаков  47  P@R 0.7566  время-протокол 0.7389  AUC 0.9316  (67 c)

  params: learning_rate=0.02, n_estimators=1500, min_child_weight=20  против  CatBoost
    разница метрики: -0.0424 (было 0.7990, стало 0.7566)
    bootstrap: среднее -0.0387, выигрывает в 1.6% пересборок, интервал 5-95%: [-0.0658, -0.0086]
    по повторам CV: -0.0343, -0.0287, -0.0317
    временной протокол: -0.0498
    [--] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [--] 4. подтверждено временным протоколом
    ВЕРДИКТ: отклонено

XGBoost — все попытки за все запуски:


,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_lambda,P@R,P@R время,сек всего,сек/обучение,принято,reg_alpha,gamma
1,500,0.048184,3,0.681758,0.503380,3,3.731784,0.7865,0.7764,14.0,0.78,False,0.006147,0.005446
3,700,0.019833,5,0.640500,0.658887,5,1.626655,0.7816,0.7757,28.1,1.56,False,0.096443,0.005309
4,800,0.025488,3,0.711369,0.612371,6,1.790667,0.7805,0.7733,23.4,1.30,False,0.042410,0.007244
2,700,0.041212,3,0.675120,0.525019,4,2.391370,0.7903,0.7716,20.6,1.14,False,0.008081,0.002575
0,1500,0.020000,6,0.800000,0.800000,20,1.000000,0.7566,0.7389,65.5,3.64,False,NaN,NaN
5,1500,0.020000,6,0.800000,0.800000,20,1.000000,0.7566,0.7389,67.1,3.73,False,NaN,NaN


In [12]:
# Optuna: автоматический перебор гиперпараметров XGBoost.
#
# Два этапа, и второй обязателен:
#   1) ПОИСК — быстрый. 5 фолдов без повторов + временной протокол, 8 обучений на попытку
#      вместо 18. Цель поиска — среднее двух протоколов: по одному случайному наверх
#      вылезут модели, цепляющиеся за период, а один временной слишком шумный.
#   2) ПРОВЕРКА — лучших пересчитываем на ПОЛНЫХ настройках и прогоняем через те же четыре
#      условия. Без этого результат поиска брать нельзя: выбор лучшего из N завышает оценку
#      сам по себе примерно на 0.041*sqrt(2*ln N), а при 30 попытках это около 0.11.

import optuna
from sklearn.model_selection import StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30
TOP_K = 4          # сколько лучших проверим честно


def _fast_eval(params):
    """Быстрая оценка кандидата: 5 фолдов + временной протокол."""
    def factory():
        return XGBClassifier(**params, device=DEVICE, tree_method="hist",
                             random_state=SEED, verbosity=0, n_jobs=-1)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.full(len(ytr), np.nan)
    for tr, va in skf.split(Xtr, ytr):
        m = factory()
        m.fit(Xtr.iloc[tr][FEATURES_BEST], ytr[tr])
        oof[va] = m.predict_proba(Xtr.iloc[va][FEATURES_BEST])[:, 1]
    return pr(ytr, oof), pr(ytr, run_time(factory, FEATURES_BEST))


def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 1500, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 30),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.5, 50.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        gamma=trial.suggest_float("gamma", 1e-3, 5.0, log=True),
    )
    a, b = _fast_eval(params)
    trial.set_user_attr("pr_random", a)
    trial.set_user_attr("pr_time", b)
    return 0.5 * a + 0.5 * b


t0 = time.time()
study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print()
print(f"поиск занял {time.time() - t0:.0f} c на {N_TRIALS} попыток")

res = pd.DataFrame([{**t.params,
                     "цель": round(t.value, 4),
                     "P@R случайный": round(t.user_attrs["pr_random"], 4),
                     "P@R время": round(t.user_attrs["pr_time"], 4)}
                    for t in study.trials if t.value is not None])
res = res.sort_values("цель", ascending=False).reset_index(drop=True)
print()
print("Топ-10 по цели поиска (быстрая оценка, ей ещё нельзя верить):")
display(res.head(10))

print()
print("Важнее любой отдельной строки — куда поиск тянет параметры:")
for c in ["max_depth", "min_child_weight", "learning_rate", "reg_lambda", "subsample"]:
    top = res.head(8)[c]
    print(f"  {c:20s} у топ-8: медиана {top.median():.4g}, "
          f"диапазон [{top.min():.4g}, {top.max():.4g}]")

  0%|          | 0/30 [00:00<?, ?it/s]


поиск занял 299 c на 30 попыток

Топ-10 по цели поиска (быстрая оценка, ей ещё нельзя верить):


,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha,gamma,цель,P@R случайный,P@R время
0,500,0.048184,3,3,0.681758,0.503380,3.731784,0.006147,0.005446,0.7728,0.7692,0.7764
1,700,0.041212,3,4,0.675120,0.525019,2.391370,0.008081,0.002575,0.7709,0.7702,0.7716
2,700,0.019833,5,5,0.640500,0.658887,1.626655,0.096443,0.005309,0.7689,0.7621,0.7757
3,800,0.025488,3,6,0.711369,0.612371,1.790667,0.042410,0.007244,0.7672,0.7612,0.7733
4,500,0.022946,4,1,0.568622,0.569629,2.865215,0.058523,0.002456,0.7606,0.7711,0.7500
5,700,0.047800,6,2,0.605148,0.704077,2.961484,1.758545,0.016990,0.7593,0.7500,0.7685
6,600,0.052029,3,6,0.707218,0.568767,2.952600,0.017361,0.001098,0.7527,0.7462,0.7591
7,400,0.079125,3,5,0.559689,0.612894,2.050152,0.044556,0.003391,0.7526,0.7575,0.7477
8,800,0.022005,6,5,0.646072,0.683181,4.084228,1.382623,0.005478,0.7513,0.7571,0.7455
9,600,0.096222,4,5,0.642477,0.508194,8.107661,0.007528,0.009504,0.7497,0.7494,0.7500



Важнее любой отдельной строки — куда поиск тянет параметры:
  max_depth            у топ-8: медиана 3, диапазон [3, 6]
  min_child_weight     у топ-8: медиана 4.5, диапазон [1, 6]
  learning_rate        у топ-8: медиана 0.04451, диапазон [0.01983, 0.07913]
  reg_lambda           у топ-8: медиана 2.628, диапазон [1.627, 3.732]
  subsample            у топ-8: медиана 0.6578, диапазон [0.5597, 0.7114]


In [13]:
# Честная проверка лучших кандидатов Optuna на полных настройках.
# try_params сам прогонит 5x3 фолда + временной протокол и применит четыре условия.

best = res.head(TOP_K)
print(f"проверяем {len(best)} лучших кандидатов, каждый примерно 30 секунд")
print()

INT_PARAMS = ("n_estimators", "max_depth", "min_child_weight")
param_cols = [c for c in res.columns if c not in ("цель", "P@R случайный", "P@R время")]

checked = []
for _, row in best.iterrows():
    kw = {c: (int(row[c]) if c in INT_PARAMS else float(row[c])) for c in param_cols}
    print("=" * 70)
    c = try_params(**kw)
    checked.append({**kw, "P@R": round(c["pr"], 4), "P@R время": round(c["pr_time"], 4)})

chk = pd.DataFrame(checked).sort_values("P@R время", ascending=False)
print()
print("Кандидаты Optuna на полных настройках, отсортировано по ВРЕМЕННОМУ протоколу:")
display(chk.reset_index(drop=True))

# Правило выбора: не argmax, а простейший из тех, что в пределах одной сигмы от лучшего.
# Простой — меньше деревьев и меньше глубина: такая модель хуже запоминает шум.
NOISE_STD = 0.041
ok = chk[chk["P@R время"] >= chk["P@R время"].max() - NOISE_STD].copy()
ok["сложность"] = ok["n_estimators"] * ok["max_depth"]
print()
print("Выбор по правилу «простейший в пределах одной сигмы»:")
display(ok.sort_values("сложность").head(1))

print()
print("Эталон для сравнения: XGBoost по умолчанию давал P@R 0.8098, временной 0.7205.")
print("Если ни один кандидат не прошёл четыре условия — тюнинг здесь ничего не даёт.")
print("Это нормальный результат, а не неудача: основной резерв был в признаках.")

проверяем 4 лучших кандидатов, каждый примерно 30 секунд



params: n_estimators=500, learning_rate=0.04818405621273927, max_depth=3, min_child_weight=3, subsample=0.6817…

params: n_estimators=500, learning_rate=0.04818405621273927, max_depth=3, min_child_weight=3, subsample=0.6817579659548862, colsample_bytree=0.5033799002037564, reg_lambda=3.7317840651021377, reg_alpha=0.00614680293036777, gamma=0.005446242658086463 признаков  47  P@R 0.7865  время-протокол 0.7764  AUC 0.9355  (14 c)

  params: n_estimators=500, learning_rate=0.04818405621273927, max_depth=3, min_child_weight=3, subsample=0.6817579659548862, colsample_bytree=0.5033799002037564, reg_lambda=3.7317840651021377, reg_alpha=0.00614680293036777, gamma=0.005446242658086463  против  XGBoost
    разница метрики: -0.0233 (было 0.8098, стало 0.7865)
    bootstrap: среднее -0.0145, выигрывает в 21.4% пересборок, интервал 5-95%: [-0.0405, +0.0200]
    по повторам CV: -0.0173, -0.0020, +0.0180
    временной протокол: +0.0559
    [--] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: о

params: n_estimators=700, learning_rate=0.04121208530000407, max_depth=3, min_child_weight=4, subsample=0.6751…

params: n_estimators=700, learning_rate=0.04121208530000407, max_depth=3, min_child_weight=4, subsample=0.675120169628478, colsample_bytree=0.5250193995919769, reg_lambda=2.391370250710841, reg_alpha=0.008081143031113007, gamma=0.002574904823143462 признаков  47  P@R 0.7903  время-протокол 0.7716  AUC 0.9349  (21 c)

  params: n_estimators=700, learning_rate=0.04121208530000407, max_depth=3, min_child_weight=4, subsample=0.675120169628478, colsample_bytree=0.5250193995919769, reg_lambda=2.391370250710841, reg_alpha=0.008081143031113007, gamma=0.002574904823143462  против  XGBoost
    разница метрики: -0.0195 (было 0.8098, стало 0.7903)
    bootstrap: среднее -0.0075, выигрывает в 31.8% пересборок, интервал 5-95%: [-0.0356, +0.0235]
    по повторам CV: -0.0163, +0.0030, +0.0123
    временной протокол: +0.0511
    [--] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: отк

...

params: n_estimators=800, learning_rate=0.02548831250941965, max_depth=3, min_child_weight=6, subsample=0.7113688372485194, colsample_bytree=0.6123706355452483, reg_lambda=1.7906667837625745, reg_alpha=0.042409859676476966, gamma=0.00724401016415003 признаков  47  P@R 0.7805  время-протокол 0.7733  AUC 0.9354  (23 c)

  params: n_estimators=800, learning_rate=0.02548831250941965, max_depth=3, min_child_weight=6, subsample=0.7113688372485194, colsample_bytree=0.6123706355452483, reg_lambda=1.7906667837625745, reg_alpha=0.042409859676476966, gamma=0.00724401016415003  против  XGBoost
    разница метрики: -0.0293 (было 0.8098, стало 0.7805)
    bootstrap: среднее -0.0219, выигрывает в 10.4% пересборок, интервал 5-95%: [-0.0484, +0.0081]
    по повторам CV: -0.0254, -0.0146, -0.0046
    временной протокол: +0.0528
    [--] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [OK] 4. подтверждено временным протоколом
    ВЕРДИКТ: о

,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha,gamma,P@R,P@R время
0,500,0.048184,3,3,0.681758,0.503380,3.731784,0.006147,0.005446,0.7865,0.7764
1,700,0.019833,5,5,0.640500,0.658887,1.626655,0.096443,0.005309,0.7816,0.7757
2,800,0.025488,3,6,0.711369,0.612371,1.790667,0.042410,0.007244,0.7805,0.7733
3,700,0.041212,3,4,0.675120,0.525019,2.391370,0.008081,0.002575,0.7903,0.7716



Выбор по правилу «простейший в пределах одной сигмы»:


,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha,gamma,P@R,P@R время,сложность
0,500,0.048184,3,3,0.681758,0.50338,3.731784,0.006147,0.005446,0.7865,0.7764,1500



Эталон для сравнения: XGBoost по умолчанию давал P@R 0.8098, временной 0.7205.
Если ни один кандидат не прошёл четыре условия — тюнинг здесь ничего не даёт.
Это нормальный результат, а не неудача: основной резерв был в признаках.


### Справочник параметров: что крутить и на что оно влияет

`try_params` принимает **любой** параметр XGBoost — в ней `params.update(kw)`, так что писать
`try_params(gamma=2)` можно уже сейчас, ничего дописывать не надо. Ниже — что вообще есть.

#### XGBoost

| параметр | что делает | куда смотреть при 8% положительных |
|---|---|---|
| `n_estimators` + `learning_rate` | сколько деревьев и насколько сильно каждое правит предыдущие | это **одна ручка**: уменьшил шаг вдвое — удвой число деревьев |
| `max_depth` | глубина; даёт до `2^depth` листьев | 3–6. При 899 ботах `depth=10` — это 1024 листа, то есть лист на каждого бота |
| `min_child_weight` | сколько наблюдений должно остаться в листе | **главная ручка при редком классе**, пробуй 5–20 |
| `gamma` | минимальный выигрыш, чтобы вообще сделать разбиение | 0–5, «не дроби без нужды» |
| `subsample` | доля строк, видимых дереву | 0.6–1.0 |
| `colsample_bytree` / `_bylevel` / `_bynode` | доля признаков на дерево / уровень / узел | 0.5–1.0 |
| `reg_lambda` | L2-штраф на величину поправок в листьях | 1–50 |
| `reg_alpha` | L1-штраф, обнуляет слабые листья | 0–10 |
| `max_delta_step` | ограничение шага; придуман ровно для дисбаланса | попробуй 1–10 |
| `scale_pos_weight` | вес положительного класса | 1 или ≈11 (это 10192/899) |
| `grow_policy="lossguide"` + `max_leaves` | растить дерево по выигрышу, как LightGBM | альтернатива обычному `depthwise` |

#### CatBoost

Устроен иначе: деревья **симметричные** — на каждом уровне все узлы делятся одним и тем же
условием. Поэтому `depth=8` у CatBoost и `max_depth=8` у XGBoost — не одно и то же, CatBoost
получается жёстче зарегуляризован по построению.

| параметр | аналог в XGBoost | что делает |
|---|---|---|
| `iterations` | `n_estimators` | число деревьев |
| `learning_rate` | `learning_rate` | шаг |
| `depth` | `max_depth` | глубина, максимум 16; на GPU до 8 работает быстрее |
| `l2_leaf_reg` | `reg_lambda` | L2-штраф, по умолчанию 3 |
| `min_data_in_leaf` | `min_child_weight` | минимум объектов в листе. **Работает только с `grow_policy="Depthwise"` или `"Lossguide"`** — при форме по умолчанию молча игнорируется, проверено |
| `random_strength` | — | шум при выборе разбиения; своя, катбустовская защита от переобучения |
| `bagging_temperature` | — | сила случайного перевзвешивания строк (при `bootstrap_type="Bayesian"`) |
| `subsample` + `bootstrap_type="Bernoulli"` | `subsample` | обычный бутстрэп по строкам |
| `rsm` | `colsample_bytree` | доля признаков. **На GPU не поддерживается** — падает с ошибкой |
| `border_count` | `max_bin` | на сколько частей режется числовой признак, по умолчанию 128 на GPU |
| `auto_class_weights="Balanced"` | `scale_pos_weight` | автоматически уравнивает классы |
| `grow_policy` | `grow_policy` | `SymmetricTree` / `Depthwise` / `Lossguide` |
| `nan_mode` | — | куда относить пропуски: `Min`, `Max`, `Forbidden` |

#### Что берём из твоего HAKATON-решения

Оттуда вытащены рабочие настройки:

```
iterations=3000..50000  early_stopping_rounds=300..500
learning_rate=0.03      depth=7..8      l2_leaf_reg=5
auto_class_weights="Balanced"   bagging_temperature=1   nan_mode="Min"
```

Что переносится: **`learning_rate=0.03`, `depth=7-8`, `l2_leaf_reg=5`, `bagging_temperature`,
`nan_mode`** — всё это про регуляризацию и не зависит от задачи. `auto_class_weights="Balanced"`
особенно интересен: у нас дисбаланс 1:11.

Что **не** переносится: `loss_function="MultiClass"` — там было несколько классов, у нас два,
нужен `Logloss`. И `cat_features` — у нас все признаки числовые, категориальных нет.

#### Про `early_stopping_rounds` — важная оговорка

В HAKATON-ноутбуке стоит `iterations=50000` с ранней остановкой по `eval_set=(X_test, y_test)`,
а потом — `model.fit(X, y, eval_set=(X_test, y_test))`, где `X` **уже содержит** `X_test`.

Это значит, что модель останавливалась по той же выборке, на которой потом мерилось качество.
Отсюда те самые 60.8% — они завышены, причём неизвестно насколько. Ровно тот случай, о котором мы
говорили в блоке про утечки: подглядывание, которое делает цифры красивее, а не хуже.

У нас ранняя остановка поэтому **не используется вообще**: число итераций фиксируется заранее.
Это честнее и проще, чем городить вложенный сплит внутри каждого фолда.

#### Про скорость

CatBoost на наших данных **на процессоре втрое быстрее, чем на видеокарте**: 8 секунд против 24.5
на одно обучение. Причина та же, что и раньше — данных слишком мало, накладные расходы на запуск
ядер съедают всю выгоду. Поэтому перебор гоняем на CPU: одна попытка ≈ 2.5 минуты вместо 7.5.

In [14]:
# CatBoost: свой перебор. Гоняем на процессоре — он тут втрое быстрее видеокарты.

CAT_DEVICE = "CPU"        # поставь "GPU", если захочешь сравнить; будет примерно втрое дольше

# Эталоны для сравнения. Каждая модель мерится со СВОИМ базовым вариантом:
# одиночный XGBoost с одиночным XGBoost, CatBoost с CatBoost. Сравнивать перебор параметров
# со смесью трёх моделей нельзя — она выигрывает за счёт усреднения, а не за счёт настроек.
TUNING_BASELINE_XGB = cand_xgb

# CatBoost-эталон пересчитываем на том же устройстве, на котором будем перебирать:
# GPU и CPU режут числовые признаки немного по-разному, и сравнение было бы нечестным.
cand_cat_base = evaluate(
    lambda: CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                               task_type=CAT_DEVICE, random_seed=SEED,
                               verbose=0, allow_writing_files=False),
    FEATURES_BEST, f"CatBoost эталон ({CAT_DEVICE})")
TUNING_BASELINE_CAT = cand_cat_base


def try_cat(**kw):
    """Одна конфигурация CatBoost против базового CatBoost. Те же четыре условия."""
    params = dict(task_type= CAT_DEVICE, iterations=500, learning_rate=0.05, depth=6,
                  l2_leaf_reg=3.0, random_strength=1.0)
    params.update(kw)

    if CAT_DEVICE == "GPU" and "rsm" in params:
        print("rsm на видеокарте не поддерживается — убираю его из этой попытки")
        params.pop("rsm")

    label = "cat: " + (", ".join(f"{k}={v}" for k, v in kw.items()) if kw else "эталон")

    def factory():
        return CatBoostClassifier(**params, random_seed=SEED,
                                  verbose=0, allow_writing_files=False)

    cand = evaluate(factory, FEATURES_BEST, label)
    ok = verdict(TUNING_BASELINE_CAT, cand)
    TRIED_CAT.append({**params, "P@R": round(cand["pr"], 4),
                      "P@R время": round(cand["pr_time"], 4),
                      "сек всего": round(cand["sec"], 1),
                      "сек/обучение": round(cand["sec"] / 18, 2), "принято": ok})
    return cand


print(f"\ntry_cat готов. Одна попытка — примерно 2.5 минуты на {CAT_DEVICE}.")
print("Результаты копятся в TRIED_CAT, отдельно от XGBoost.")

CatBoost эталон (CPU):   0%|          | 0/15 [00:00<?, ?it/s]

CatBoost эталон (CPU)        признаков  47  P@R 0.7995  время-протокол 0.7757  AUC 0.9359  (29 c)

try_cat готов. Одна попытка — примерно 2.5 минуты на CPU.
Результаты копятся в TRIED_CAT, отдельно от XGBoost.


In [16]:
# Что пробовать у CatBoost. Раскомментируй ПО ОДНОЙ строке — каждая идёт около 2.5 минут.

# --- настройки из твоего HAKATON-решения (топ-3 из 40) ---
# try_cat(learning_rate=0.03, iterations=1500, depth=8, l2_leaf_reg=5)
try_cat( task_type = "CPU", learning_rate=0.03, iterations=3000, depth=7, l2_leaf_reg=10,auto_class_weights="Balanced", early_stopping_rounds=100, boosting_type = "Ordered")
# try_cat(nan_mode="Min")

# --- дисбаланс: у нас 1 бот на 11 людей ---
# try_cat(auto_class_weights="Balanced")

# --- регуляризация: при редком классе обычно помогает сильнее всего ---
# try_cat(depth=4)
# try_cat(grow_policy="Depthwise", min_data_in_leaf=20)   # без Depthwise параметр молча игнорируется
# try_cat(l2_leaf_reg=10)
# try_cat(random_strength=3)

# --- случайность по строкам и признакам ---
# try_cat(bootstrap_type="Bernoulli", subsample=0.8)
# try_cat(rsm=0.8)                      # только на CPU
# try_cat(bagging_temperature=5)

# --- форма дерева: по умолчанию симметричная, но можно как у XGBoost/LightGBM ---
# try_cat(grow_policy="Depthwise", depth=6)
# try_cat(grow_policy="Lossguide", max_leaves=64)

# --- точность разбиения числовых признаков ---
# try_cat(border_count=254)

for nm, journal in [("XGBoost", TRIED_XGB), ("CatBoost", TRIED_CAT)]:
    if journal:
        t = pd.DataFrame(journal)
        front = [c for c in ["P@R", "P@R время", "сек всего", "сек/обучение", "принято"]
                 if c in t.columns]
        t = t[front + [c for c in t.columns if c not in front]]
        print(f"\n=== {nm}: {len(journal)} попыток, отсортировано по ВРЕМЕННОМУ протоколу ===")
        display(t.sort_values("P@R время", ascending=False).reset_index(drop=True))
    else:
        print(f"\n=== {nm}: попыток пока нет ===")


cat: task_type=CPU, learning_rate=0.03, iterations=3000, depth=7, l2_leaf_reg=10, auto_class_weights=Balanced,…

cat: task_type=CPU, learning_rate=0.03, iterations=3000, depth=7, l2_leaf_reg=10, auto_class_weights=Balanced, early_stopping_rounds=100, boosting_type=Ordered признаков  47  P@R 0.7326  время-протокол 0.7433  AUC 0.9269  (1424 c)

  cat: task_type=CPU, learning_rate=0.03, iterations=3000, depth=7, l2_leaf_reg=10, auto_class_weights=Balanced, early_stopping_rounds=100, boosting_type=Ordered  против  CatBoost эталон (CPU)
    разница метрики: -0.0669 (было 0.7995, стало 0.7326)
    bootstrap: среднее -0.0687, выигрывает в 0.0% пересборок, интервал 5-95%: [-0.1035, -0.0357]
    по повторам CV: -0.1161, -0.1044, -0.1052
    временной протокол: -0.0324
    [--] 1. средняя разница > 0
    [--] 2. выигрывает в >= 90% пересборок
    [--] 3. знак одинаков на всех повторах
    [--] 4. подтверждено временным протоколом
    ВЕРДИКТ: отклонено

=== XGBoost: 5 попыток, отсортировано по ВРЕМЕННОМУ протоколу ===


,P@R,P@R время,сек всего,сек/обучение,принято,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_lambda,reg_alpha,gamma
0,0.7865,0.7764,14.0,0.78,False,500,0.048184,3,0.681758,0.503380,3,3.731784,0.006147,0.005446
1,0.7816,0.7757,28.1,1.56,False,700,0.019833,5,0.640500,0.658887,5,1.626655,0.096443,0.005309
2,0.7805,0.7733,23.4,1.30,False,800,0.025488,3,0.711369,0.612371,6,1.790667,0.042410,0.007244
3,0.7903,0.7716,20.6,1.14,False,700,0.041212,3,0.675120,0.525019,4,2.391370,0.008081,0.002575
4,0.7566,0.7389,65.5,3.64,False,1500,0.020000,6,0.800000,0.800000,20,1.000000,NaN,NaN



=== CatBoost: 1 попыток, отсортировано по ВРЕМЕННОМУ протоколу ===


,P@R,P@R время,сек всего,сек/обучение,принято,task_type,iterations,learning_rate,depth,l2_leaf_reg,random_strength,auto_class_weights,early_stopping_rounds,boosting_type
0,0.7326,0.7433,1424.3,79.13,False,CPU,3000,0.03,7,10,1.0,Balanced,100,Ordered


In [ ]:
# Автоматический случайный поиск. ВЫКЛЮЧЕН: включай, только если ручной перебор себя исчерпал.

RUN_SEARCH = False
N_CONFIGS = 20

PARAM_SPACE = {
    "learning_rate": [0.02, 0.03, 0.05, 0.08],
    "n_estimators": [300, 500, 800, 1500],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0, 20.0],
}

if RUN_SEARCH:
    rng = np.random.default_rng(SEED)
    results = []
    for i in tqdm(range(N_CONFIGS), desc="случайный поиск"):
        cfg = {k: v[rng.integers(len(v))] for k, v in PARAM_SPACE.items()}
        c = try_params(**cfg)
        results.append({**cfg, "pr": c["pr"], "pr_time": c["pr_time"]})
    res = pd.DataFrame(results).sort_values("pr", ascending=False)
    display(res)

    # Берём НЕ лучшую, а простейшую из тех, что в пределах одной сигмы от лучшей.
    # Простая — меньше деревьев и меньше глубина: такая хуже запоминает шум.
    NOISE_STD = 0.041
    ok = res[res["pr"] >= res["pr"].max() - NOISE_STD].copy()
    ok["сложность"] = ok["n_estimators"] * ok["max_depth"]
    print("\nвыбор по правилу «простейшая в пределах сигмы»:")
    display(ok.sort_values("сложность").head(1))
else:
    print("автоматический поиск выключен (RUN_SEARCH = False).")
    print(f"при 20 конфигурациях завышение от отбора по шуму составит около "
          f"{0.041 * np.sqrt(2 * np.log(20)):.3f} — держи это в голове.")

### Выбор чемпиона: почему нельзя смотреть только на случайный протокол

В ячейке 15 чемпион выбирается по случайному протоколу. Прогон показал, что это **неверно**, и вот
почему.

Реальный тест — это дни 20–26, а обучаемся мы на днях 6–19. То есть задача формулируется как
«учись на прошлом, предсказывай будущее». **Временной протокол воспроизводит именно её**, а
случайный отвечает на другой вопрос: «угадай отложенную куку из того же периода».

В твоём прогоне два протокола выстроили модели почти в обратном порядке:

| модель | P@R случайный | P@R временной |
|---|---|---|
| XGBoost | 0.8098 (1 место) | 0.7205 (**последнее**) |
| LightGBM | 0.8036 | 0.7576 |
| CatBoost | 0.7990 (последнее) | 0.7886 (**1 место**) |
| смесь трёх | 0.7955 | 0.7806 |
| XGBoost ×5 seed | 0.7878 | 0.7545 |

Разрыв у XGBoost между протоколами — **0.089**, у CatBoost — всего 0.010. Это значит, что
XGBoost сильнее цепляется за особенности конкретного периода: на перемешанных данных он их
использует, на сдвинутых во времени — теряет.

Adversarial validation говорила, что сдвига распределений нет (AUC 0.51). Значит дело не в том,
что test другой, а в том, **сколько данных достаётся на обучение**: во временных фолдах модель
учится на 8–12 днях вместо 80% всех дней. Кто устойчивее к урезанию обучающей выборки, тот и
выигрывает на временном протоколе.

Ниже считается суммарный ранг по обоим протоколам. Модель, которая хороша на обоих, надёжнее
той, что блестит на одном и проваливается на другом, — особенно когда лидерборда нет и
переиграть уже не получится.

In [17]:
# Чемпион с учётом обоих протоколов.

rows = []
for k, v in CANDIDATES.items():
    rows.append({"кандидат": k,
                 "P@R случайный": round(v["pr"], 4),
                 "P@R временной": round(v["pr_time"], 4)})
b = pd.DataFrame(rows)
b["ранг случайный"] = b["P@R случайный"].rank(ascending=False).astype(int)
b["ранг временной"] = b["P@R временной"].rank(ascending=False).astype(int)
b["сумма рангов"] = b["ранг случайный"] + b["ранг временной"]
b["разрыв между протоколами"] = (b["P@R случайный"] - b["P@R временной"]).round(4)
b = b.sort_values(["сумма рангов", "P@R временной"], ascending=[True, False]).reset_index(drop=True)

print("Кандидаты по обоим протоколам сразу:")
display(b)

best_time = b.sort_values("P@R временной", ascending=False).iloc[0]
best_sum = b.iloc[0]
print(f"\nлучший по временному протоколу : {best_time['кандидат']}  ({best_time['P@R временной']:.4f})")
print(f"лучший по сумме рангов         : {best_sum['кандидат']}  "
      f"(случайный {best_sum['P@R случайный']:.4f}, временной {best_sum['P@R временной']:.4f})")

# Большой разрыв между протоколами — тревожный признак: модель держится за особенности периода.
worst_gap = b.sort_values("разрыв между протоколами", ascending=False).iloc[0]
print(f"\nсамый большой разрыв между протоколами у {worst_gap['кандидат']}: "
      f"{worst_gap['разрыв между протоколами']:+.4f}")
print("чем разрыв меньше, тем устойчивее модель к тому, что тест сдвинут во времени")

# Итоговый выбор: берём по сумме рангов. Если хочешь перебить руками — присвой CHAMPION_FINAL сам.
CHAMPION_FINAL = CANDIDATES[best_sum["кандидат"]]
print(f"\nЧЕМПИОН ДЛЯ ПЕРЕНОСА В solution.ipynb: {CHAMPION_FINAL['name']}")

Кандидаты по обоим протоколам сразу:


,кандидат,P@R случайный,P@R временной,ранг случайный,ранг временной,сумма рангов,разрыв между протоколами
0,CatBoost,0.7990,0.7886,3,1,4,0.0104
1,LightGBM,0.8036,0.7576,2,3,5,0.0460
2,смесь XGB+LGBM+Cat (ранги),0.7955,0.7806,4,2,6,0.0149
3,XGBoost,0.8098,0.7205,1,5,6,0.0893
4,XGBoost x5 seed,0.7878,0.7545,5,4,9,0.0333



лучший по временному протоколу : CatBoost  (0.7886)
лучший по сумме рангов         : CatBoost  (случайный 0.7990, временной 0.7886)

самый большой разрыв между протоколами у XGBoost: +0.0893
чем разрыв меньше, тем устойчивее модель к тому, что тест сдвинут во времени

ЧЕМПИОН ДЛЯ ПЕРЕНОСА В solution.ipynb: CatBoost


## Итог: что переносить в solution.ipynb

In [18]:
# Сводка всех экспериментов и что делать дальше.

summary = pd.DataFrame(REGISTRY).drop_duplicates(subset="name", keep="last")
summary = summary.sort_values("pr", ascending=False).reset_index(drop=True)
summary.columns = ["эксперимент", "признаков", "P@R случайный", "P@R временной", "ROC-AUC", "сек"]
print("Все эксперименты этого ноутбука:")
display(summary.round(4))

print("\n" + "=" * 72)
print("ЧТО ПЕРЕНОСИТЬ В solution.ipynb")
print("=" * 72)
print(f"1. Принятые группы признаков: {accepted if accepted else 'ни одной'}")
print(f"   Отклонённые (не переносим): {rejected if rejected else 'нет'}")
print(f"2. Итоговый набор: {len(FEATURES_BEST)} признаков")
print(f"3. Модель: {CHAMPION_FINAL['name']}")
print(f"4. Ожидаемое качество: P@R = {CHAMPION_FINAL['pr']:.4f} "
      f"(на старте было 0.5004)")
print(f"   прирост: {CHAMPION_FINAL['pr'] - 0.5004:+.4f}")
print("\nОтклонённые группы НЕ переносим, даже если по абсолютной цифре они выглядели")
print("чуть лучше: они не прошли проверку на устойчивость, а лидерборда, который бы")
print("нас поправил, нет. Это и есть вся дисциплина этого ноутбука.")
print("\nСписок признаков для переноса:")
print(FEATURES_BEST)

Все эксперименты этого ноутбука:


,эксперимент,признаков,P@R случайный,P@R временной,ROC-AUC,сек
0,+ uaplat,66,0.8108,0.7733,0.9332,25.1339
1,+ pointer,47,0.8098,0.7205,0.9326,23.3659
2,+ sequence,54,0.8056,0.7615,0.9321,24.9108
3,LightGBM,47,0.8036,0.7576,0.9347,8.0453
4,CatBoost эталон (CPU),47,0.7995,0.7757,0.9359,28.7178
5,CatBoost,47,0.7990,0.7886,0.9367,346.3080
6,смесь XGB+LGBM+Cat (ранги),47,0.7955,0.7806,0.9364,0.0000
7,"params: n_estimators=700, learning_rate=0.0412...",47,0.7903,0.7716,0.9349,20.6039
8,XGBoost x5 seed,47,0.7878,0.7545,0.9331,122.1901
9,"params: n_estimators=500, learning_rate=0.0481...",47,0.7865,0.7764,0.9355,13.9791



ЧТО ПЕРЕНОСИТЬ В solution.ipynb
1. Принятые группы признаков: ['content', 'pointer']
   Отклонённые (не переносим): ['rhythm', 'navquery', 'sequence', 'uaplat']
2. Итоговый набор: 47 признаков
3. Модель: CatBoost
4. Ожидаемое качество: P@R = 0.7990 (на старте было 0.5004)
   прирост: +0.2986

Отклонённые группы НЕ переносим, даже если по абсолютной цифре они выглядели
чуть лучше: они не прошли проверку на устойчивость, а лидерборда, который бы
нас поправил, нет. Это и есть вся дисциплина этого ноутбука.

Список признаков для переноса:
['n_events', 'n_items_uniq', 'n_cat_uniq', 'n_loc_uniq', 'n_query_uniq', 'events_per_hour', 'cnt_search_results_view', 'cnt_item_view', 'cnt_photo_swipe', 'cnt_seller_page_view', 'cnt_contact_phone_show', 'cnt_contact_chat_open', 'cnt_contact_message_sent', 'cnt_favorite_add', 'cnt_login', 'dt_median', 'dt_min', 'dt_std', 'dt_frac_lt_1s', 'span_seconds', 'n_active_hours', 'photo_per_view', 'contact_per_view', 'view_per_search', 'max_search_page', 'mean_s

In [22]:
print(np.shape(train))

(11091, 5)
